# 01 — Préparation des données (Transfermarkt)

Cette première partie configure les sources, les charge, vérifie leur schéma, nettoie les types et prépare les informations Transfermarkt à la date `REF_DATE`. L'export FBref n'est pas encore traité. `appearances.csv` est chargé et contrôlé, mais ne sert ni aux minutes ni aux statistiques per-90.

Les données brutes ne sont jamais modifiées. Aucune valeur n'est fabriquée ou imputée. La sortie finale et le rapprochement FBref seront ajoutés dans une étape ultérieure.

In [1]:
import os
from pathlib import Path

import pandas as pd

RANDOM_STATE = 42
REF_DATE = pd.Timestamp(os.getenv("SCOUTHUB_REF_DATE", "2025-06-30")).normalize()
VALUATION_MAX_AGE_DAYS = int(os.getenv("VALUATION_MAX_AGE_DAYS", "365"))
COMPETITION_CODES = {"GB1", "ES1", "IT1", "L1", "FR1"}

def _find_ml_root():
    cwd = Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "ml", base / "scouthub" / "ml"))
    for candidate in candidates:
        if (candidate / "requirements.txt").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Racine ml/ introuvable depuis le répertoire courant. "
        "Définissez SCOUTHUB_ML_ROOT."
    )

ML_ROOT = Path(os.environ["SCOUTHUB_ML_ROOT"]).expanduser() if "SCOUTHUB_ML_ROOT" in os.environ else _find_ml_root()
ML_ROOT = ML_ROOT.resolve()
if not (ML_ROOT / "requirements.txt").is_file():
    raise FileNotFoundError(f"SCOUTHUB_ML_ROOT ne pointe pas vers ml/: {ML_ROOT}")

RAW_DIR = Path(os.getenv("SCOUTHUB_ML_RAW_DIR", str(ML_ROOT / "data" / "raw"))).expanduser()
if not RAW_DIR.is_absolute():
    RAW_DIR = ML_ROOT / RAW_DIR
RAW_DIR = RAW_DIR.resolve()

SOURCE_FILES = {
    "players": os.getenv("TM_PLAYERS_FILE", "players.csv"),
    "player_valuations": os.getenv("TM_PLAYER_VALUATIONS_FILE", "player_valuations.csv"),
    "appearances": os.getenv("TM_APPEARANCES_FILE", "appearances.csv"),
    "clubs": os.getenv("TM_CLUBS_FILE", "clubs.csv"),
    "competitions": os.getenv("TM_COMPETITIONS_FILE", "competitions.csv"),
    "games": os.getenv("TM_GAMES_FILE", "games.csv"),
}

if pd.isna(REF_DATE):
    raise ValueError("SCOUTHUB_REF_DATE doit être une date valide, par exemple 2025-06-30.")
if VALUATION_MAX_AGE_DAYS < 0:
    raise ValueError("VALUATION_MAX_AGE_DAYS doit être un entier positif ou nul.")

print(f"ML_ROOT      : {ML_ROOT}")
print(f"RAW_DIR      : {RAW_DIR}")
print(f"REF_DATE     : {REF_DATE.date()}")
print(f"Âge maximal de valorisation : {VALUATION_MAX_AGE_DAYS} jours")
print(f"Compétitions : {', '.join(sorted(COMPETITION_CODES))}")
print(f"random_state : {RANDOM_STATE} (aucun tirage aléatoire dans cette étape)")

ML_ROOT      : C:\Projet_dev_haqiq\scouthub\ml
RAW_DIR      : C:\Projet_dev_haqiq\scouthub\ml\data\raw
REF_DATE     : 2025-06-30
Âge maximal de valorisation : 365 jours
Compétitions : ES1, FR1, GB1, IT1, L1
random_state : 42 (aucun tirage aléatoire dans cette étape)


In [ ]:
# Étape 2 — Chargement des cinq tables Transfermarkt et contrôles de schéma.
def _source_path(configured_path):
    path = Path(configured_path).expanduser()
    return path if path.is_absolute() else RAW_DIR / path

source_paths = {name: _source_path(filename) for name, filename in SOURCE_FILES.items()}
missing_files = [str(path) for path in source_paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Tables Transfermarkt manquantes :\n- " + "\n- ".join(missing_files)
        + "\nPlacez les CSV dans ml/data/raw/ ou configurez SCOUTHUB_ML_RAW_DIR "
        "et les variables TM_*_FILE."
    )

tables = {
    name: pd.read_csv(path, low_memory=False)
    for name, path in source_paths.items()
}

REQUIRED_COLUMNS = {
    "players": {
        "player_id", "name", "date_of_birth", "position", "sub_position", "foot",
        "height_in_cm", "country_of_birth", "country_of_citizenship",
        "contract_expiration_date",
    },
    "player_valuations": {
        "player_id", "date", "market_value_in_eur", "current_club_id",
        "player_club_domestic_competition_id",
    },
    "appearances": {"player_id"},
    "clubs": {"club_id", "name"},
    "competitions": {"competition_id", "competition_code"},
    "games": {"competition_id", "season", "date", "home_club_id", "away_club_id"},
}

schema_errors = []
for table_name, required in REQUIRED_COLUMNS.items():
    missing_columns = sorted(required - set(tables[table_name].columns))
    if missing_columns:
        schema_errors.append(f"{table_name}: {', '.join(missing_columns)}")
if schema_errors:
    raise ValueError("Colonnes Transfermarkt requises absentes :\n- " + "\n- ".join(schema_errors))

source_inventory = pd.DataFrame(
    [
        {"table": name, "path": str(source_paths[name]), "rows": len(frame), "columns": len(frame.columns)}
        for name, frame in tables.items()
    ]
)
display(source_inventory)
for name, frame in tables.items():
    print(f"{name}: {list(frame.columns)}")

In [ ]:
# Étape 3 — Nettoyage des types. Les valeurs non interprétables restent manquantes
# et sont comptabilisées ; aucune imputation n'est effectuée.
type_issues = []

def _clean_id(series):
    cleaned = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    return cleaned.replace("", pd.NA)

for table_name, columns in {
    "players": ["player_id"],
    "player_valuations": ["player_id", "current_club_id", "player_club_domestic_competition_id"],
    "appearances": ["player_id"],
    "clubs": ["club_id"],
    "competitions": ["competition_id"],
    "games": ["competition_id", "home_club_id", "away_club_id"],
}.items():
    for column in columns:
        tables[table_name][column] = _clean_id(tables[table_name][column])

for table_name, column in {
    "players": "date_of_birth",
    "player_valuations": "date",
    "games": "date",
}.items():
    original = tables[table_name][column]
    parsed = pd.to_datetime(original, errors="coerce")
    invalid = original.notna() & parsed.isna()
    type_issues.append({"table": table_name, "column": column, "invalid_values": int(invalid.sum())})
    tables[table_name][column] = parsed

original = tables["games"]["season"]
parsed = pd.to_numeric(original, errors="coerce")
invalid = original.notna() & parsed.isna()
type_issues.append({"table": "games", "column": "season", "invalid_values": int(invalid.sum())})
tables["games"]["season"] = parsed.astype("Int64")

for column in ["contract_expiration_date"]:
    original = tables["players"][column]
    parsed = pd.to_datetime(original, errors="coerce")
    invalid = original.notna() & parsed.isna()
    type_issues.append({"table": "players", "column": column, "invalid_values": int(invalid.sum())})
    tables["players"][column] = parsed

for table_name, column in {
    "players": "height_in_cm",
    "player_valuations": "market_value_in_eur",
}.items():
    original = tables[table_name][column]
    parsed = pd.to_numeric(original, errors="coerce")
    invalid = original.notna() & parsed.isna()
    type_issues.append({"table": table_name, "column": column, "invalid_values": int(invalid.sum())})
    tables[table_name][column] = parsed

for table_name, columns in {
    "players": ["name", "position", "sub_position", "foot", "country_of_birth", "country_of_citizenship"],
    "clubs": ["name"],
    "competitions": ["competition_code"],
}.items():
    for column in columns:
        tables[table_name][column] = tables[table_name][column].astype("string").str.strip()
        tables[table_name][column] = tables[table_name][column].replace("", pd.NA)

display(pd.DataFrame(type_issues))
for table_name in ("players", "player_valuations", "clubs", "competitions"):
    print(f"\nValeurs manquantes — {table_name}")
    display(tables[table_name].isna().sum().rename("missing_count").to_frame())

print("appearances.csv est chargé pour contrôle, mais n'est pas utilisé pour les minutes ou les statistiques.")

In [ ]:
# Étape 4 — Déduplication contrôlée et préparation Transfermarkt à REF_DATE.
# Les doublons exacts sont retirés. Des lignes contradictoires pour une même clé
# provoquent une erreur plutôt qu'un choix arbitraire.
def _report_loss(stage, before, after):
    print(f"{stage}: {before:,} -> {after:,} lignes; perdues: {before - after:,}")

def _unique_on(frame, key_columns, label):
    before = len(frame)
    result = frame.drop_duplicates().copy()
    _report_loss(f"{label} — doublons exacts", before, len(result))
    duplicate_mask = result.duplicated(key_columns, keep=False)
    if duplicate_mask.any():
        conflicting = result.loc[duplicate_mask].groupby(key_columns, dropna=False).nunique(dropna=False)
        conflicting_keys = conflicting.index[conflicting.gt(1).any(axis=1)]
        if len(conflicting_keys):
            examples = list(conflicting_keys[:5])
            raise ValueError(f"{label}: valeurs contradictoires pour {key_columns}; exemples: {examples}")
        before = len(result)
        result = result.drop_duplicates(subset=key_columns, keep="first").copy()
        _report_loss(f"{label} — répétitions de clé identiques", before, len(result))
    return result

player_columns = [
    "player_id", "name", "date_of_birth", "position", "sub_position", "foot",
    "height_in_cm", "country_of_birth", "country_of_citizenship",
    "contract_expiration_date",
]
players_tm = tables["players"][player_columns].copy()
before = len(players_tm)
players_tm = players_tm.loc[players_tm["player_id"].notna()].copy()
_report_loss("players — identifiant manquant", before, len(players_tm))
players_tm = _unique_on(players_tm, ["player_id"], "players")

valuation_columns = [
    "player_id", "date", "market_value_in_eur", "current_club_id",
    "player_club_domestic_competition_id",
]
valuations_tm = tables["player_valuations"][valuation_columns].copy()
before = len(valuations_tm)
valuations_tm = valuations_tm.loc[valuations_tm["player_id"].notna() & valuations_tm["date"].notna()].copy()
_report_loss("player_valuations — identifiant/date manquant ou date invalide", before, len(valuations_tm))
valuation_start_date = REF_DATE - pd.Timedelta(days=VALUATION_MAX_AGE_DAYS)
before = len(valuations_tm)
valuations_tm = valuations_tm.loc[
    valuations_tm["date"].between(valuation_start_date, REF_DATE, inclusive="both")
].copy()
_report_loss(
    f"player_valuations — hors fenêtre [{valuation_start_date.date()}, {REF_DATE.date()}]",
    before,
    len(valuations_tm),
)
valuations_tm = _unique_on(valuations_tm, ["player_id", "date"], "player_valuations (player_id, date)")

before = len(valuations_tm)
valuations_tm = valuations_tm.sort_values(["player_id", "date"], kind="stable")
latest_valuations = valuations_tm.drop_duplicates(subset=["player_id"], keep="last").copy()
_report_loss("player_valuations — conservation de la dernière valeur à REF_DATE", before, len(latest_valuations))

clubs_tm = tables["clubs"][ ["club_id", "name"] ].copy()
clubs_tm = clubs_tm.loc[clubs_tm["club_id"].notna()].copy()
clubs_tm = _unique_on(clubs_tm, ["club_id"], "clubs")
clubs_tm = clubs_tm.rename(columns={"club_id": "current_club_id", "name": "club"})

competitions_tm = tables["competitions"][ ["competition_id", "competition_code"] ].copy()
competitions_tm = competitions_tm.loc[competitions_tm["competition_id"].notna()].copy()
competitions_tm = _unique_on(competitions_tm, ["competition_id"], "competitions")
competitions_tm = competitions_tm.rename(
    columns={
        "competition_id": "player_club_domestic_competition_id",
        "competition_code": "competition_slug",
    }
)

before = len(latest_valuations)
transfermarkt_asof = latest_valuations.merge(
    players_tm, on="player_id", how="inner", validate="one_to_one"
)
_report_loss("Rapprochement interne — valorisation sans fiche joueur", before, len(transfermarkt_asof))
before = len(transfermarkt_asof)
transfermarkt_asof = transfermarkt_asof.merge(
    clubs_tm, on="current_club_id", how="left", validate="many_to_one"
)
_report_loss("Jointure du club courant", before, len(transfermarkt_asof))
before = len(transfermarkt_asof)
transfermarkt_asof = transfermarkt_asof.merge(
    competitions_tm, on="player_club_domestic_competition_id", how="left", validate="many_to_one"
)
_report_loss("Jointure de la compétition", before, len(transfermarkt_asof))
games_2024 = tables["games"].loc[
    tables["games"]["season"].eq(2024)
    & tables["games"]["competition_id"].isin(COMPETITION_CODES),
    ["competition_id", "date", "home_club_id", "away_club_id"],
].copy()
season_summary = (
    games_2024.groupby("competition_id")
    .agg(matches=("competition_id", "size"), first_date=("date", "min"), last_date=("date", "max"))
    .reindex(sorted(COMPETITION_CODES))
)
print("Saison Transfermarkt 2024 — matchs et dates observées :")
display(season_summary)

club_league_2024 = pd.concat(
    [
        games_2024[["competition_id", "home_club_id"]].rename(
            columns={"competition_id": "league", "home_club_id": "club_id"}
        ),
        games_2024[["competition_id", "away_club_id"]].rename(
            columns={"competition_id": "league", "away_club_id": "club_id"}
        ),
    ],
    ignore_index=True,
).dropna(subset=["club_id"]).drop_duplicates()
if club_league_2024["club_id"].duplicated().any():
    raise ValueError("Un club apparaît dans plusieurs compétitions cibles en saison 2024.")

club_counts = (
    club_league_2024.groupby("league")["club_id"]
    .nunique()
    .reindex(sorted(COMPETITION_CODES), fill_value=0)
    .rename("n_clubs")
    .reset_index()
)
print("Clubs distincts par compétition en saison 2024 :")
display(club_counts)
expected_club_counts = {"GB1": 20, "ES1": 20, "IT1": 20, "L1": 18, "FR1": 18}
observed_club_counts = club_counts.set_index("league")["n_clubs"].to_dict()
if observed_club_counts != expected_club_counts:
    raise ValueError(
        f"Effectifs de clubs inattendus pour la saison 2024 : {observed_club_counts}; "
        f"attendus : {expected_club_counts}."
    )
assert club_league_2024["club_id"].nunique() == 96, "Le périmètre doit contenir 96 clubs distincts."

before = len(transfermarkt_asof)
club_to_league = club_league_2024.set_index("club_id")["league"]
transfermarkt_asof["league"] = transfermarkt_asof["current_club_id"].map(club_to_league)
transfermarkt_asof = transfermarkt_asof.loc[transfermarkt_asof["league"].notna()].copy()
_report_loss("Filtre par club présent dans une compétition cible en saison 2024", before, len(transfermarkt_asof))
players_by_league = (
    transfermarkt_asof.groupby("league")["player_id"]
    .nunique()
    .reindex(sorted(COMPETITION_CODES), fill_value=0)
    .rename("n_players")
    .reset_index()
)
print("Joueurs retenus par ligue :")
display(players_by_league)

birth_date = transfermarkt_asof["date_of_birth"]
has_birth_date = birth_date.notna()
age_values = (
    REF_DATE.year
    - birth_date.dt.year
    - (
        (birth_date.dt.month > REF_DATE.month)
        | ((birth_date.dt.month == REF_DATE.month) & (birth_date.dt.day > REF_DATE.day))
    ).astype("int64")
)
transfermarkt_asof["age"] = age_values.where(has_birth_date).astype("Int64")
transfermarkt_asof = transfermarkt_asof.rename(
    columns={
        "date_of_birth": "birth_date",
        "height_in_cm": "height",
        "country_of_citizenship": "nationality",
        "contract_expiration_date": "contract_expiration",
        "market_value_in_eur": "market_value_eur",
    }
)
transfermarkt_asof = transfermarkt_asof[
    [
        "player_id", "name", "birth_date", "age", "position", "sub_position", "foot",
        "height", "nationality", "country_of_birth", "current_club_id", "club", "league",
        "contract_expiration", "market_value_eur",
    ]
].rename(columns={"current_club_id": "club_id"})

assert transfermarkt_asof["player_id"].is_unique, "Plusieurs lignes Transfermarkt par joueur après préparation."
assert transfermarkt_asof["league"].isin(COMPETITION_CODES).all(), "Compétition hors périmètre détectée."
assert latest_valuations["date"].le(REF_DATE).all(), "Une valorisation postérieure à REF_DATE a été conservée."

print(f"Joueurs Transfermarkt préparés : {len(transfermarkt_asof):,}")
print(f"Joueurs sans date de naissance (âge manquant) : {transfermarkt_asof['birth_date'].isna().sum():,}")
print(f"Joueurs avec market_value_eur manquant : {transfermarkt_asof['market_value_eur'].isna().sum():,}")
print("Aucune sortie CSV n'est écrite dans cette étape ; le traitement FBref est différé.")
display(transfermarkt_asof.head())

In [ ]:
league_summary = (
    transfermarkt_asof.groupby("league")
    .agg(n_players=("player_id", "nunique"), n_clubs=("club_id", "nunique"))
    .reindex(sorted(COMPETITION_CODES), fill_value=0)
    .rename_axis("league")
    .reset_index()
)
display(league_summary)

In [ ]:
print(transfermarkt_asof["club_id"].nunique())
print(transfermarkt_asof.groupby("league").size())
print(transfermarkt_asof.groupby("club")["player_id"].size().describe())

# Préparation FBref

Chargement et contrôle de l’export `players_data-2024_2025.csv`, conservation des lignes joueur-club, filtrage configurable à 450 minutes et calcul des statistiques par 90 minutes. Les valeurs invalides ou aberrantes sont signalées, jamais imputées ni corrigées. Le rapprochement FBref et toute écriture de fichier restent hors de cette étape.

In [5]:
# Étape C1 — Chargement FBref, schéma, types et cardinalité joueur-club.
import numpy as np

FBREF_FILE = RAW_DIR / "players_data-2024_2025.csv"
fbref = pd.read_csv(FBREF_FILE, low_memory=False)

FBREF_REQUIRED_COLUMNS = [
    "Player", "Born", "Squad", "Comp", "Pos", "Min", "90s",
    "Gls", "Ast", "xG", "PrgP", "PrgC", "Tkl", "Int", "Succ",
]
missing_fbref_columns = [column for column in FBREF_REQUIRED_COLUMNS if column not in fbref.columns]
if missing_fbref_columns:
    raise ValueError(
        f"Colonnes FBref exactes manquantes dans {FBREF_FILE}: {missing_fbref_columns}"
    )

print(f"Fichier FBref : {FBREF_FILE}")
print(f"Dimensions : {fbref.shape[0]:,} lignes × {fbref.shape[1]:,} colonnes")
print("Valeurs manquantes — colonnes demandées :")
display(fbref[FBREF_REQUIRED_COLUMNS].isna().sum().rename("missing_count").to_frame())

fbref_squad_counts = (
    fbref.groupby("Comp")["Squad"]
    .nunique()
    .rename("n_clubs")
    .sort_index()
    .to_frame()
)
print("Nombre de clubs distincts par compétition :")
display(fbref_squad_counts)
expected_fbref_squad_counts = {
    "eng Premier League": 20,
    "es La Liga": 20,
    "it Serie A": 20,
    "fr Ligue 1": 18,
    "de Bundesliga": 18,
}
observed_fbref_squad_counts = fbref_squad_counts["n_clubs"].to_dict()
if observed_fbref_squad_counts != expected_fbref_squad_counts:
    raise ValueError(
        f"Nombre de clubs FBref inattendu: {observed_fbref_squad_counts}; "
        f"attendu: {expected_fbref_squad_counts}."
    )
assert fbref_squad_counts["n_clubs"].sum() == 96, "Le périmètre FBref doit contenir 96 clubs."

# Les valeurs non numériques / années de naissance non entières deviennent manquantes;
# les nombres d'invalides sont conservés dans le rapport, sans imputation.
fbref_type_issues = []
born_original = fbref["Born"].astype("string").str.strip().replace("", pd.NA)
born_numeric = pd.to_numeric(born_original, errors="coerce")
born_non_integer = born_numeric.notna() & born_numeric.mod(1).ne(0)
born_invalid = born_original.notna() & (born_numeric.isna() | born_non_integer)
fbref_type_issues.append({"column": "Born", "invalid_values": int(born_invalid.sum())})
fbref["Born"] = born_numeric.mask(born_non_integer).astype("Int64")

fbref_numeric_columns = ["Min", "90s", "Gls", "Ast", "xG", "PrgP", "PrgC", "Tkl", "Int", "Succ"]
for column in fbref_numeric_columns:
    original = fbref[column].astype("string").str.strip().replace("", pd.NA)
    parsed = pd.to_numeric(original, errors="coerce")
    invalid = original.notna() & parsed.isna()
    fbref_type_issues.append({"column": column, "invalid_values": int(invalid.sum())})
    fbref[column] = parsed

print("Valeurs non interprétables pendant la conversion numérique (converties en manquantes) :")
display(pd.DataFrame(fbref_type_issues))

fbref_player_club_counts = fbref.groupby(["Player", "Born"], dropna=False)["Squad"].nunique()
fbref_transferred_players = int(fbref_player_club_counts.gt(1).sum())
print(f"Joueurs présents dans plusieurs clubs (lignes conservées) : {fbref_transferred_players:,}")
fbref_duplicate_mask = fbref.duplicated(subset=["Player", "Born", "Squad"], keep=False)
fbref_duplicate_count = int(fbref_duplicate_mask.sum())
print(f"Lignes impliquées dans un doublon (Player, Born, Squad) : {fbref_duplicate_count:,}")
if fbref_duplicate_mask.any():
    display(fbref.loc[fbref_duplicate_mask, ["Player", "Born", "Squad", "Comp"]].head(20))
    raise ValueError("Doublon détecté sur la clé (Player, Born, Squad).")


Fichier FBref : C:\Projet_dev_haqiq\scouthub\ml\data\raw\players_data-2024_2025.csv
Dimensions : 2,854 lignes ? 267 colonnes
Valeurs manquantes ? colonnes demand?es :


,missing_count
Player,0
Born,8
Squad,0
Comp,0
Pos,0
Min,0
90s,0
Gls,0
Ast,0
xG,0


Nombre de clubs distincts par comp?tition :


,n_clubs
Comp,
de Bundesliga,18
eng Premier League,20
es La Liga,20
fr Ligue 1,18
it Serie A,20


Valeurs non interpr?tables pendant la conversion num?rique (converties en manquantes) :


,column,invalid_values
0,Born,0
1,Min,0
2,90s,0
3,Gls,0
4,Ast,0
5,xG,0
6,PrgP,0
7,PrgC,0
8,Tkl,0
9,Int,0


Joueurs pr?sents dans plusieurs clubs (lignes conserv?es) : 147
Lignes impliqu?es dans un doublon (Player, Born, Squad) : 0


In [3]:
# Étape C2 — Seuil de minutes, statistiques per-90 et indicateur gardien.
MIN_MINUTES = 450
if MIN_MINUTES < 0:
    raise ValueError("MIN_MINUTES doit être positif ou nul.")

fbref_zero_minute_rows = int(fbref["Min"].eq(0).sum())
fbref_before_filter = len(fbref)
fbref_clean = fbref.loc[fbref["Min"].ge(MIN_MINUTES)].copy()
fbref_lost_rows = fbref_before_filter - len(fbref_clean)
print(f"Seuil : Min >= {MIN_MINUTES:,}")
print(f"Lignes perdues : {fbref_before_filter:,} -> {len(fbref_clean):,} ; perdues : {fbref_lost_rows:,}")
print(f"Lignes à 0 minute avant filtre : {fbref_zero_minute_rows:,}")

# Un taux avec 90s nul est indéfini; on garde la ligne et signale le dénominateur nul.
fbref_zero_90s_rows = int(fbref_clean["90s"].eq(0).sum())
print(f"Lignes retenues avec 90s = 0 (per-90 indéfinis) : {fbref_zero_90s_rows:,}")
fbref_per90_columns = {
    "Gls": "goals_p90",
    "Ast": "assists_p90",
    "xG": "xg_p90",
    "PrgP": "prog_passes_p90",
    "PrgC": "prog_carries_p90",
    "Tkl": "tackles_p90",
    "Int": "interceptions_p90",
    "Succ": "dribbles_p90",
}
fbref_denominator = fbref_clean["90s"].mask(fbref_clean["90s"].eq(0))
for source_column, rate_column in fbref_per90_columns.items():
    fbref_clean[rate_column] = fbref_clean[source_column].div(fbref_denominator)

fbref_clean = fbref_clean.rename(columns={"Min": "minutes"})
fbref_clean["is_goalkeeper"] = fbref_clean["Pos"].eq("GK")
print(f"Gardiens conservés : {int(fbref_clean['is_goalkeeper'].sum()):,}")
print(f"Joueurs-lignes conservés dans fbref_clean : {len(fbref_clean):,}")


Seuil : Min >= 450
Lignes perdues : 2,854 -> 1,982 ; perdues : 872
Lignes ? 0 minute avant filtre : 0
Lignes retenues avec 90s = 0 (per-90 ind?finis) : 0
Gardiens conserv?s : 155
Joueurs-lignes conserv?s dans fbref_clean : 1,982


In [4]:
# Étape C3 — Résumé et signalement des valeurs aberrantes, sans correction.
print("Statistiques descriptives des colonnes per-90 :")
display(fbref_clean[list(fbref_per90_columns.values())].describe())
print(f"Minutes égales à 0 dans fbref_clean : {int(fbref_clean['minutes'].eq(0).sum()):,}")

fbref_outlier_rows = []
fbref_any_extreme = pd.Series(False, index=fbref_clean.index)
for rate_column in fbref_per90_columns.values():
    values = fbref_clean[rate_column]
    finite_values = values.loc[np.isfinite(values)]
    negative_mask = values.lt(0).fillna(False)
    non_finite_mask = values.notna() & ~np.isfinite(values)
    if finite_values.empty:
        lower_fence = np.nan
        upper_fence = np.nan
        tukey_extreme_mask = pd.Series(False, index=fbref_clean.index)
    else:
        q1 = finite_values.quantile(0.25)
        q3 = finite_values.quantile(0.75)
        iqr = q3 - q1
        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr
        tukey_extreme_mask = values.lt(lower_fence).fillna(False) | values.gt(upper_fence).fillna(False)
    extreme_mask = non_finite_mask | tukey_extreme_mask
    fbref_any_extreme |= extreme_mask
    fbref_outlier_rows.append({
        "per90_column": rate_column,
        "negative_values": int(negative_mask.sum()),
        "extreme_values_Tukey_1.5IQR": int(extreme_mask.sum()),
        "lower_fence": lower_fence,
        "upper_fence": upper_fence,
        "non_finite_values": int(non_finite_mask.sum()),
    })

fbref_outlier_summary = pd.DataFrame(fbref_outlier_rows)
print("Valeurs négatives et extrêmes (seuil Tukey = 1,5 × IQR; valeurs conservées) :")
display(fbref_outlier_summary)
if fbref_any_extreme.any():
    print("Exemples de lignes comportant au moins une valeur per-90 extrême :")
    display(
        fbref_clean.loc[
            fbref_any_extreme,
            ["Player", "Born", "Squad", "minutes", *fbref_per90_columns.values()],
        ].head(20)
    )
else:
    print("Aucune valeur per-90 extrême détectée selon les bornes Tukey.")


Statistiques descriptives des colonnes per-90 :


,goals_p90,assists_p90,xg_p90,prog_passes_p90,prog_carries_p90,tackles_p90,interceptions_p90,dribbles_p90
count,1982.0,1982.0,1982.0,1982.0,1982.0,1982.0,1982.0,1982.0
mean,0.126003,0.088866,0.13081,3.340025,1.567739,1.531824,0.740869,0.679627
std,0.168777,0.103986,0.152856,1.91271,1.328452,0.857275,0.514519,0.64722
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.027988,2.093483,0.622571,0.982659,0.329128,0.201512
50%,0.064935,0.062112,0.070302,3.204774,1.264837,1.517428,0.696258,0.514649
75%,0.186047,0.144144,0.185906,4.427766,2.221329,2.075472,1.091693,0.967119
max,1.149425,0.689655,1.152542,11.455696,12.678571,5.132743,2.621359,6.369048


Minutes ?gales ? 0 dans fbref_clean : 0
Valeurs n?gatives et extr?mes (seuil Tukey = 1,5 ? IQR; valeurs conserv?es) :


,per90_column,negative_values,extreme_values_Tukey_1.5IQR,lower_fence,upper_fence,non_finite_values
0,goals_p90,0,92,-0.279070,0.465116,0
1,assists_p90,0,44,-0.216216,0.360361,0
2,xg_p90,0,118,-0.208888,0.422782,0
3,prog_passes_p90,0,39,-1.407941,7.929190,0
4,prog_carries_p90,0,67,-1.775565,4.619466,0
5,tackles_p90,0,23,-0.656560,3.714691,0
6,interceptions_p90,0,9,-0.814719,2.235540,0
7,dribbles_p90,0,71,-0.946899,2.115530,0


Exemples de lignes comportant au moins une valeur per-90 extr?me :


,Player,Born,Squad,minutes,goals_p90,assists_p90,xg_p90,prog_passes_p90,prog_carries_p90,tackles_p90,interceptions_p90,dribbles_p90
15,Zakaria Aboukhlal,2000,Toulouse,1901,0.331754,0.094787,0.483412,1.042654,1.611374,1.042654,0.28436,0.758294
18,Tammy Abraham,1997,Milan,1183,0.229008,0.305344,0.541985,1.984733,0.610687,1.29771,0.152672,0.305344
25,Tyler Adams,1999,Bournemouth,1965,0.0,0.137615,0.073394,3.486239,0.642202,3.807339,1.46789,0.183486
29,Karim Adeyemi,2002,Dortmund,1433,0.440252,0.377358,0.345912,1.761006,4.27673,1.257862,0.440252,2.515723
32,Yacine Adli,2000,Fiorentina,1219,0.296296,0.444444,0.074074,7.62963,1.777778,2.37037,0.444444,0.888889
37,Oladapo Afolayan,1998,St. Pauli,1639,0.164835,0.054945,0.17033,1.758242,2.912088,0.934066,0.43956,2.637363
60,Ludovic Ajorque,1994,Brest,2495,0.469314,0.072202,0.375451,1.480144,0.361011,0.938628,0.216606,0.288809
64,Ilias Akhomach,2004,Villarreal,584,0.153846,0.0,0.092308,1.846154,4.615385,1.846154,0.461538,2.923077
66,Maghnes Akliouche,2002,Monaco,2406,0.187266,0.374532,0.344569,6.479401,4.756554,1.910112,0.561798,1.872659
72,Chuba Akpom,1995,Lille,639,0.422535,0.140845,0.704225,0.985915,0.84507,1.126761,0.0,0.56338
